# Bổ sung lớp tham chiếu: 5 PC đầu và embedding g7 (bản 2 của bộ dữ liệu)

Chạy SAU `s2_HF_LOP_THAM_CHIEU` (bộ dữ liệu `lopmaybay/haiphong-lop-tham-chieu` đã có s2tc, s2sw, pca, lulc_ctx, lulc_ctxts, hai bộ điểm). Notebook này **chỉ thêm**, không ghi đè ảnh cũ; `manifest.json` cũ được giữ thành `manifest_v1.json`.

| lớp mới | nội dung | cỡ ước tính |
|---|---|---|
| `pc1` ... `pc5` | từng PC của PCA chuỗi năm, ảnh xám 8 bit, **cùng thang cho mọi năm, đối xứng quanh 0** (trang tô đỏ = dương, xanh = âm, trắng = 0) | ~40 MB/PC/năm, tổng ~2 GB |
| `pc5d` | dữ liệu 5 PC (int16) + NFILL, lưới UTM gốc: trang **tái dựng đường NDVI, MNDWI ở điểm BẤT KỲ** (kể cả điểm thêm tay) với k = 5 | ~0.4 GB/năm |
| `diem/*_duong_cong_k5.json` | đường mùa vụ tính sẵn cho hai bộ điểm, **k = 5** (thay k = 11: các PC sau PC5 là mây và vệt quỹ đạo) | vài MB |
| `g7`, `g7b` | embedding g7 (`g7_chi_recon`, hệ số chuẩn hoá CHUNG như chuỗi g7c), chiếu lên 6 thành phần chính chung cho mọi năm: RGB thành phần 1-2-3 và 4-5-6 | ~0.3 GB/lớp/năm |
| `diem/emb_g7_*.parquet`, `diem/pc5_*.parquet` | embedding 64 chiều và 5 PC tại 900 điểm E0 và các điểm ứng viên, mọi năm: dùng thẳng để huấn luyện bộ phân loại nhiều lớp sau khi gán nhãn | vài MB |
| `emb/g7_phep_chieu.json` | phép chiếu 64 → 6 (tâm, thành phần, kéo giãn): tái lập được | nhỏ |

**Cách tính embedding giống hệt bản đồ g7c**: chia ô 1024 từ góc ảnh, lề = `enc.pad`, đệm đối xứng, fp16. Cổng kiểm tra: màu của ảnh tại 900 điểm E0 phải trùng (±1 mức) với embedding tính riêng tại điểm bằng `s2_e6.embed_points_tiled` (hàm đã qua cổng R0 của E6).

**Runtime**: GPU (A100, L4 hoặc T4) cho phần embedding; phần PC chạy CPU cũng được. Chạy tiếp được nếu bị ngắt (bỏ qua tệp đã có). Thời gian: PC ~2 giờ; embedding một lượt mỗi năm, các bước gối đầu, cỡ 1 phút/năm trên A100.

**Trước khi chạy**: tải lên HP_modules bản mới của `s2_hf_lop.py` (1.2) và `LAY_MAU_DA_NAM.html` (1.1, có lớp xám và bảng màu); ô 0 sẽ hỏi nếu còn bản cũ.

In [ ]:
# ============ 0. THAM SỐ ============
HF_REPO = "lopmaybay/haiphong-lop-tham-chieu"
NAM = list(range(2017, 2027))
K5 = 5                         # chỉ 5 PC đầu (các PC sau là mây, vệt quỹ đạo)
LAM = dict(pc_xam=True, pc_du_lieu=True, duong_k5=True, pc_diem=True, emb=True, emb_diem=True)
# encoder: đường dẫn tính từ HP_3class_v1 hoặc chỉ tên tệp (sẽ tự tìm trong HP_3class_v1)
EMB_CAU_HINH = [
    dict(ten="g7", lam=True, ckpt="thesis_v1/timkiem/encoders/enc_g7_chi_recon.pt",
         he_so_chung="thesis_v1/series_g7_chung/he_so_chung.json",
         ghi_chu="g7_chi_recon, chuẩn hoá CHUNG cho mọi năm (như chuỗi g7c)"),
    dict(ten="dil_MY", lam=False, ckpt="enc_dil_MY.pt", he_so_chung=None,
         ghi_chu="encoder đa niên dil_MY, chuẩn hoá theo từng ảnh (như chuỗi dil_MY)"),
    dict(ten="dil_AB", lam=False, ckpt="enc_dil_AB.pt", he_so_chung=None,
         ghi_chu="encoder dil_AB của bài v7, chuẩn hoá theo từng ảnh"),
]
N_TP = 6                       # số thành phần chính của embedding: 2 ảnh RGB (1-2-3 và 4-5-6)
NAM_KHOP = (2019, 2021, 2023, 2025)   # các năm lấy mẫu để khớp phép chiếu chung
CHO_PHEP_CPU = False           # embedding toàn tỉnh trên CPU rất chậm; chỉ bật khi thật cần
GUI_LEN = False                # True: đẩy các tệp MỚI lên Hugging Face ở ô cuối
CAP_NHAT_SPACE = False         # True: cập nhật index.html của Space (nếu đã tạo ở notebook trước)
SPACE = "lopmaybay/lay-mau-hai-phong"

!pip install -q geopandas rasterio huggingface_hub
from google.colab import drive; drive.mount("/content/drive")
import os, sys, glob, json, re, time, shutil, hashlib
import numpy as np, pandas as pd, geopandas as gpd, rasterio
D = "/content/drive/MyDrive"; MOD = f"{D}/HP_modules"
OUT3 = f"{D}/HP_3class_v1"; MY = f"{OUT3}/multiyear"; TH = f"{OUT3}/thesis_v1"
DRIVE_MY = f"{D}/HP_multiyear"; KQ_TS = f"{D}/HP_PCA_TS_KETQUA/S2"
HF_DIR = f"{D}/HP_HF_LOP_THAM_CHIEU"
assert os.path.exists(f"{HF_DIR}/manifest.json"), "chưa có bộ dữ liệu bản 1: chạy s2_HF_LOP_THAM_CHIEU trước"
TMP = "/content/tmp_hf"; SRC = "/content/src_hf"; os.makedirs(TMP, exist_ok=True); os.makedirs(SRC, exist_ok=True)


def phien_ban(p, mau):
    # đọc số phiên bản trong tệp (vd __version__ = "1.1") -> tuple, không có -> (0,)
    if not os.path.exists(p):
        return (0,)
    m = re.search(mau, open(p, encoding="utf-8", errors="ignore").read())
    return tuple(int(v) for v in m.group(1).split(".")) if m else (0,)


NEED = ["s2_hf_lop.py", "s2_multiyear.py", "s2_e6.py", "s2_fast.py", "s2_classify.py", "s2_pixlab.py",
        "pixel_embed_v4_matryoshka.py", "LAY_MAU_DA_NAM.html"]
thieu = [m for m in NEED if not os.path.exists(f"{MOD}/{m}")]
if phien_ban(f"{MOD}/s2_hf_lop.py", r'__version__ = "([\d.]+)"') < (1, 1):
    thieu.append("s2_hf_lop.py (bản 1.1)")
if phien_ban(f"{MOD}/LAY_MAU_DA_NAM.html", r'const VERSION = "([\d.]+)"') < (1, 1):
    thieu.append("LAY_MAU_DA_NAM.html (bản 1.1)")
if thieu:
    print("Tải lên HP_modules:", thieu); from google.colab import files
    for k in files.upload(): shutil.copy(k, MOD)
if MOD not in sys.path: sys.path.insert(0, MOD)
for m in ["s2_hf_lop", "s2_multiyear", "s2_e6", "s2_fast", "s2_classify", "s2_pixlab"]: sys.modules.pop(m, None)
import s2_hf_lop as H, s2_multiyear as M
assert H.__version__ >= "1.1", f"s2_hf_lop {H.__version__}: cần bản 1.1"
assert H.self_test(verbose=False) == 0, "s2_hf_lop tự kiểm tra KHÔNG đạt"
MAN0 = json.load(open(f"{HF_DIR}/manifest.json"))
print("s2_hf_lop", H.__version__, "| bộ dữ liệu bản 1:", [L["id"] for L in MAN0["layers"]])

In [ ]:
# ============ 1. NGUỒN, LƯỚI, HAI BỘ ĐIỂM, KÉO GIÃN 5 PC ============
cho = (glob.glob(f"{D}/HP_PCA_TS_CHUNG/he_so_S2.json")
       + glob.glob("/content/drive/.shortcut-targets-by-id/*/HP_PCA_TS_CHUNG/he_so_S2.json"))
HS = json.load(open(cho[0])); MA = HS["ma"]
assert HS["k"] >= K5
HE5 = {k: HS[k] for k in ["mean", "std", "W", "bang", "he_so_nhan"] if k in HS}
HE5["W"] = HS["W"][:K5]
TS_BY = {y: f"{KQ_TS}/HP_PCA_S2_{y}_{MA}" + ("_tam" if y == 2026 else "") + ".tif" for y in NAM}
RAS = {y: f"{DRIVE_MY}/S2_HP_{y}.tif" for y in NAM}
RAS.update({y: f"{DRIVE_MY}/S2_HP_{y}_v2.tif" for y in (2017, 2018)})     # ảnh đã vá, như chuỗi bản đồ
for ten, d in [("S2", RAS), ("PCA", TS_BY)]:
    print(f"  {ten:4s}: có {[y for y, p in d.items() if os.path.exists(p)]}")

bnd, _ = M.read_boundary(f"{D}/TÀI LIỆU KHOÁ HỌC VIỄN THÁM/DỮ LIỆU/Hải Phòng (phường xã) - 34.zip")
TF, WID, HEI, BND = M.boundary_grid(bnd, res=10, pad_m=200)
khung = (TF.c, TF.f + HEI * TF.e, TF.c + WID * TF.a, TF.f)
GRID = H.grid_3857(khung, "EPSG:32648", res=10.0, pad_m=300)          # đúng lưới của bản 1
print(f"lưới 3857: {GRID[1]} × {GRID[2]}")


def may(p):
    q = f"{SRC}/{hashlib.md5(p.encode()).hexdigest()[:8]}_{os.path.basename(p)}"
    if not (os.path.exists(q) and os.path.getsize(q) == os.path.getsize(p)):
        shutil.copyfile(p, q)
    return q


def bo(p):
    q = f"{SRC}/{hashlib.md5(p.encode()).hexdigest()[:8]}_{os.path.basename(p)}"
    if os.path.exists(q):
        os.remove(q)


def dung(lop, y, fn):
    out = f"{HF_DIR}/{lop}/{lop}_{y}.tif"
    if os.path.exists(out):
        return out
    os.makedirs(os.path.dirname(out), exist_ok=True)
    loc = f"{TMP}/{lop}_{y}.tif"
    fn(loc)
    shutil.copyfile(loc, out); os.remove(loc)
    return out


BO_DIEM = {"E0_900": pd.read_csv(f"{HF_DIR}/diem/E0_900.csv"), "ung_vien": pd.read_csv(f"{HF_DIR}/diem/ung_vien.csv")}
print("hai bộ điểm:", {k: len(v) for k, v in BO_DIEM.items()})

# kéo giãn CỐ ĐỊNH, ĐỐI XỨNG quanh 0 cho từng PC (năm tham chiếu 2023, như lớp pca của bản 1)
lo_, hi_ = H.percentiles(may(TS_BY[2023]), list(range(1, K5 + 1)), nodata=H.TS_NODATA)
KG5 = [float(max(abs(a), abs(b))) for a, b in zip(lo_, hi_)]
print("kéo giãn PC1-5 (± PC × 100):", [round(v) for v in KG5])

In [ ]:
# ============ 2. PC1-5: ẢNH XÁM, ẢNH DỮ LIỆU 5 PC, ĐƯỜNG MÙA VỤ k = 5, 5 PC TẠI ĐIỂM ============
import s2_e6 as E6
os.makedirs(f"{HF_DIR}/diem", exist_ok=True)
DUONG5 = {}
for ten, d in BO_DIEM.items():
    p = f"{HF_DIR}/diem/{ten}_duong_cong_k5.json"
    DUONG5[ten] = json.load(open(p)) if os.path.exists(p) else dict(
        id=list(d["id"]), ky=HS.get("ky", [[1, 2], [3, 4], [5, 6], [7, 8], [9, 10], [11, 12]]), k=K5, nam={})
PC_DIEM = {ten: (pd.read_parquet(f"{HF_DIR}/diem/pc5_{ten}.parquet")
                 if os.path.exists(f"{HF_DIR}/diem/pc5_{ten}.parquet") else pd.DataFrame()) for ten in BO_DIEM}
t0 = time.time()
for y in NAM:
    if not os.path.exists(TS_BY[y]):
        continue
    print(f"== {y}")
    if LAM["pc_xam"]:
        for j in range(1, K5 + 1):
            dung(f"pc{j}", y, lambda o, j=j: H.gray_cog(may(TS_BY[y]), j, -KG5[j - 1], KG5[j - 1], o, GRID,
                                                        nodata=H.TS_NODATA, tmp_dir=TMP))
    if LAM["pc_du_lieu"]:
        dung("pc5d", y, lambda o: H.pc_cog(may(TS_BY[y]), o, k=K5, tmp_dir=TMP))
    for ten, d in BO_DIEM.items():
        if LAM["duong_k5"] and str(y) not in DUONG5[ten]["nam"]:
            DUONG5[ten]["nam"].update(H.pc_curves(d.x_utm.values, d.y_utm.values, [y], {y: may(TS_BY[y])}, HE5, k=K5))
            json.dump(DUONG5[ten], open(f"{HF_DIR}/diem/{ten}_duong_cong_k5.json", "w"))
        if LAM["pc_diem"] and not (len(PC_DIEM[ten]) and (PC_DIEM[ten].year == y).any()):
            t_ = E6.sample_ts(d.x_utm.values, d.y_utm.values, np.full(len(d), y), {y: may(TS_BY[y])}, K5)
            t_.insert(0, "year", y); t_.insert(0, "id", d["id"].values)
            PC_DIEM[ten] = pd.concat([PC_DIEM[ten], t_], ignore_index=True)
            PC_DIEM[ten].to_parquet(f"{HF_DIR}/diem/pc5_{ten}.parquet", index=False)
    bo(TS_BY[y])
    print(f"   {(time.time() - t0) / 60:.0f} phút")
for lop in [f"pc{j}" for j in range(1, K5 + 1)] + ["pc5d"]:
    fs = glob.glob(f"{HF_DIR}/{lop}/*.tif")
    if fs:
        print(f"  {lop:5s}: {len(fs)} tệp, {sum(os.path.getsize(f) for f in fs) / 1e9:.2f} GB")

In [ ]:
# ============ 3. EMBEDDING NHANH (GPU): MỘT LƯỢT MỖI NĂM, CHIẾU TRÊN GPU, CÁC BƯỚC GỐI ĐẦU ============
# Nhanh hơn ô cũ (4 phút/năm) nhờ:
#  (1) MỘT lượt encoder mỗi năm cho cả ảnh chiếu lẫn vector 64 chiều tại hai bộ điểm (ô cũ chạy encoder 3 lần);
#  (2) gộp các ô 1024 cùng cỡ thành LÔ trên GPU; chiếu 64 -> 6 và lượng tử hoá ngay trên GPU, chỉ chép uint8 về;
#  (3) nhiều luồng đọc ô trong khi GPU chạy;
#  (4) chép ảnh S2 các năm sau từ Drive TRONG KHI đang tính năm này; dựng hai COG (g7, g7b) trong MỘT lượt nắn,
#      nhiều luồng, ở nền, gối đầu với năm sau.
# Chạy tiếp được: năm đã có đủ g7, g7b hợp lệ và đủ điểm thì bỏ qua; tệp chép dở (bị ngắt) được phát hiện, làm lại.
LO = 8          # số ô 1024 mỗi lô GPU (A100 80 GB: 8-16 thoải mái). Cổng không đạt thì đặt LO = 1
N_DOC = 4       # số luồng đọc ô
N_CHEP = 2      # số năm chép trước từ Drive
N_COG = 2       # số năm dựng COG song song ở nền
sys.modules.pop("s2_hf_lop", None)
if phien_ban(f"{MOD}/s2_hf_lop.py", r'__version__ = "([\d.]+)"') < (1, 2):
    print("Tải lên HP_modules: s2_hf_lop.py (bản 1.2)"); from google.colab import files
    for k in files.upload(): shutil.copy(k, MOD)
    sys.modules.pop("s2_hf_lop", None)
import s2_hf_lop as H
assert H.__version__ >= "1.2", f"s2_hf_lop {H.__version__}: cần bản 1.2"
import torch, s2_pixlab as P, s2_fast as FA, s2_e6 as E6
from concurrent.futures import ThreadPoolExecutor
DEV = "cuda" if torch.cuda.is_available() else "cpu"
if LAM["emb"] and DEV == "cpu" and not CHO_PHEP_CPU:
    LAM["emb"] = False
    print("KHÔNG có GPU: bỏ phần embedding (đổi runtime sang GPU rồi chạy lại ô này)")


def tim(p):
    if not p:
        return None
    for q in (p, f"{OUT3}/{p}"):
        if os.path.exists(q):
            return q
    c = sorted(glob.glob(f"{OUT3}/**/{os.path.basename(p)}", recursive=True))
    return c[0] if c else None


def nap_encoder(ten, ckpt):
    st = torch.load(ckpt, map_location="cpu", weights_only=False)
    cfg = st.get("cfg", {}) or {}
    net = P.build_arch(cfg.get("arch", "dilated_v4"), 10, int(cfg.get("dim", 64)),
                       width=int(cfg.get("width", 48)), dilations=tuple(cfg.get("dilations", (1, 2, 4))),
                       depth=int(cfg.get("depth", 3)))
    net.load_state_dict(st["state_dict"])
    enc = P.DenseEncoder(ten, net, st["mean"], st["std"], int(cfg.get("dim", 64))).to(DEV)
    enc.pad = st.get("pad", 12); enc.net.eval()
    return enc


MASKS = {}


def mat_na(sp):
    with rasterio.open(sp) as d_:
        sig = (d_.height, d_.width, tuple(np.round(d_.transform[:6], 3)))
    if sig not in MASKS:
        MASKS[sig] = FA.boundary_mask(sp, bnd)
    return MASKS[sig]


def luu_parquet(df_, p):
    df_.to_parquet(p + ".part", index=False); os.replace(p + ".part", p)


CONG = {}
for cf in (EMB_CAU_HINH if LAM["emb"] else []):
    if not cf["lam"]:
        continue
    ten = cf["ten"]; ck = tim(cf["ckpt"])
    if ck is None:
        print(f"[{ten}] không thấy {cf['ckpt']}: bỏ qua"); continue
    try:
        enc = nap_encoder(ten, ck)
    except Exception as e:
        print(f"[{ten}] không nạp được encoder {ck}: {type(e).__name__}: {e}"); continue
    mom = None
    if cf.get("he_so_chung"):
        hp = tim(cf["he_so_chung"]); assert hp, f"[{ten}] không thấy {cf['he_so_chung']}"
        hs_ = json.load(open(hp)); mom = (np.array(hs_["mean"], np.float32), np.array(hs_["std"], np.float32))
    print(f"[{ten}] {ck} | dim {enc.dim}, lề {enc.pad}, chuẩn hoá {'chung' if mom else 'theo từng ảnh'}, lô {LO}")

    # (a) phép chiếu 64 -> N_TP CHUNG cho mọi năm (đã có từ lần chạy trước thì dùng lại)
    os.makedirs(f"{HF_DIR}/emb", exist_ok=True)
    p_pr = f"{HF_DIR}/emb/{ten}_phep_chieu.json"
    if os.path.exists(p_pr):
        PR = json.load(open(p_pr))
    else:
        Z = []
        for y in NAM_KHOP:
            sp = may(RAS[y])
            Z.append(H.embed_sample(sp, H.torch_embed_fn(enc, sp, mom, DEV), enc.pad, mask=mat_na(sp),
                                    n_tiles=6, per_tile=5000, tile=1024, seed=y))
        PR = H.fit_proj(np.concatenate(Z), n=N_TP)
        PR.update(encoder=ten, ckpt=os.path.basename(ck), chuan_hoa="chung" if mom else "theo từng ảnh",
                  nam_khop=list(NAM_KHOP), tile=1024, pad=int(enc.pad), ghi_chu=cf["ghi_chu"])
        json.dump(PR, open(p_pr, "w"))
    mu_, C_ = np.asarray(PR["mu"], np.float32), np.asarray(PR["comps"], np.float32)
    print(f"   tỉ lệ phương sai {N_TP} thành phần: {np.round(PR['ti_le_phuong_sai'], 3).tolist()}")

    # (b) các năm còn thiếu
    EMB_DIEM = {b: (pd.read_parquet(f"{HF_DIR}/diem/emb_{ten}_{b}.parquet")
                    if os.path.exists(f"{HF_DIR}/diem/emb_{ten}_{b}.parquet") else pd.DataFrame()) for b in BO_DIEM}
    ra = lambda y: (f"{HF_DIR}/{ten}/{ten}_{y}.tif", f"{HF_DIR}/{ten}b/{ten}b_{y}.tif")
    xong_anh = lambda y: all(os.path.exists(o) and H.hop_le(o) for o in ra(y))
    xong_diem = lambda y: (not LAM["emb_diem"]) or all(len(v) and (v.year == y).any() for v in EMB_DIEM.values())
    nam_lam = [y for y in NAM if os.path.exists(RAS[y]) and not (xong_anh(y) and xong_diem(y))]
    print(f"   còn phải làm: {nam_lam}")
    p_cong = f"{HF_DIR}/emb/{ten}_cong_nhanh.json"
    if os.path.exists(p_cong):
        CONG[ten] = json.load(open(p_cong))
    pool_chep, pool_cog = ThreadPoolExecutor(N_CHEP), ThreadPoolExecutor(N_COG)
    f_chep, viec_cog, t0 = {}, [], time.time()


    def lam_cog(y, u8, ten=ten):
        o1, o2 = ra(y)
        if not xong_anh(y):
            a1, a2 = f"{TMP}/{ten}_{y}.tif", f"{TMP}/{ten}b_{y}.tif"
            H.rgb_cog_nhom(u8, [([1, 2, 3], a1), ([4, 5, 6], a2)], GRID, tmp_dir=TMP, verbose=False)
            H.chep_an_toan(a1, o1); H.chep_an_toan(a2, o2); os.remove(a1); os.remove(a2)
        os.remove(u8)
        return y, time.time()


    try:
        for i, y in enumerate(nam_lam):
            for y_ in nam_lam[i:i + N_CHEP + 1]:                 # chép trước các năm sau
                if y_ not in f_chep:
                    f_chep[y_] = pool_chep.submit(may, RAS[y_])
            sp = f_chep.pop(y).result()
            u8 = f"{TMP}/{ten}_{y}_u8.tif"
            pts = {b: (d.x_utm.values, d.y_utm.values) for b, d in BO_DIEM.items()} if LAM["emb_diem"] else None
            res = H.embed_pass(sp, H.torch_proj_fn(enc, sp, PR, mom, DEV), enc.pad, N_TP, u8, tile=1024,
                               mask=mat_na(sp), points=pts, batch=LO, n_doc=N_DOC)
            if ten not in CONG:
                # CỔNG của đường nhanh (lô, chiếu trên GPU): so với embed_points_tiled (từng ô một, như bản đồ g7c)
                d = BO_DIEM["E0_900"]
                zr = E6.embed_points_tiled(enc, sp, d.x_utm.values, d.y_utm.values, moments=mom, tile=1024,
                                           device=DEV).to_numpy(np.float32)
                q_ref = H.quantize(((zr - mu_) @ C_.T).T, PR["lo"], PR["hi"]).T.astype(int)
                q_ras = H.sample_bands(u8, d.x_utm.values, d.y_utm.values).astype(int)
                ok = (q_ref > 0).all(1) & (q_ras > 0).all(1)
                t_anh = float((np.abs(q_ref[ok] - q_ras[ok]) <= 1).all(1).mean()) if ok.any() else 0.0
                t_diem = float("nan")
                if res:
                    zp = res["E0_900"].to_numpy(np.float32); ok2 = np.isfinite(zr).all(1) & np.isfinite(zp).all(1)
                    t_diem = float(np.abs(zr[ok2] - zp[ok2]).max() / max(np.abs(zr[ok2]).max(), 1e-9))
                CONG[ten] = dict(trung_anh=t_anh, lech_diem_tuong_doi=t_diem, n_diem=int(ok.sum()), nam=y, lo=LO)
                json.dump(CONG[ten], open(p_cong, "w"))
                print(f"   CỔNG {ten} (đường nhanh, lô {LO}): ảnh trùng {t_anh:.4f} ở {int(ok.sum())} điểm E0; "
                      f"vector tại điểm lệch tương đối tối đa {t_diem:.1e}")
                assert t_anh >= 0.99 and not (t_diem > 0.01), "đường nhanh không khớp: đặt LO = 1 rồi chạy lại ô này"
            if res:
                for b, d in BO_DIEM.items():
                    if not (len(EMB_DIEM[b]) and (EMB_DIEM[b].year == y).any()):
                        ze = res[b].copy(); ze.insert(0, "year", y); ze.insert(0, "id", d["id"].values)
                        EMB_DIEM[b] = pd.concat([EMB_DIEM[b], ze], ignore_index=True)
                        luu_parquet(EMB_DIEM[b], f"{HF_DIR}/diem/emb_{ten}_{b}.parquet")
            bo(RAS[y])                                           # phép chiếu đã khớp: không cần giữ ảnh S2
            viec_cog.append(pool_cog.submit(lam_cog, y, u8))
            print(f"   {ten} {y}: xong phần GPU, {(time.time() - t0) / 60:.1f} phút", flush=True)
        for f_ in viec_cog:
            y_, t_ = f_.result()
            print(f"   {ten} {y_}: COG xong, {(t_ - t0) / 60:.1f} phút")
    finally:
        pool_chep.shutdown(wait=True); pool_cog.shutdown(wait=True)
    del enc
    if DEV == "cuda":
        torch.cuda.empty_cache()
for lop in [c["ten"] + s for c in EMB_CAU_HINH for s in ("", "b")]:
    fs = glob.glob(f"{HF_DIR}/{lop}/*.tif")
    if fs:
        print(f"  {lop:8s}: {len(fs)} tệp, {sum(os.path.getsize(f) for f in fs) / 1e9:.2f} GB")

In [ ]:
# ============ 4. MANIFEST BẢN 2, README, TRANG HTML ============
def co(lop):
    return sorted(int(re.findall(r"(20\d\d)", os.path.basename(p))[0]) for p in glob.glob(f"{HF_DIR}/{lop}/{lop}_*.tif"))


if not os.path.exists(f"{HF_DIR}/manifest_v1.json"):
    shutil.copyfile(f"{HF_DIR}/manifest.json", f"{HF_DIR}/manifest_v1.json")      # giữ bản 1
MAN = json.load(open(f"{HF_DIR}/manifest_v1.json"))
moi = []
for j in range(1, K5 + 1):
    if co(f"pc{j}"):
        moi.append(dict(id=f"pc{j}", ten=f"PC{j} chuỗi năm (đỏ +, xanh −, trắng 0)", kieu="xam",
                        bang_mau_lien_tuc="rdbu", duong_dan=f"pc{j}/pc{j}_{{y}}.tif", nam=co(f"pc{j}"),
                        keo_gian=[-KG5[j - 1], KG5[j - 1]]))
for cf in EMB_CAU_HINH:
    for suf, tp in [("", "1-2-3"), ("b", "4-5-6")]:
        lop = cf["ten"] + suf
        if co(lop):
            moi.append(dict(id=lop, ten=f"Embedding {cf['ten']}, thành phần {tp}", kieu="rgb",
                            duong_dan=f"{lop}/{lop}_{{y}}.tif", nam=co(lop),
                            phep_chieu=f"emb/{cf['ten']}_phep_chieu.json"))
ids = {L["id"] for L in moi}
MAN["layers"] = [L for L in MAN["layers"] if L["id"] not in ids] + moi
if co("pc5d"):
    MAN["pc"] = dict(duong_dan="pc5d/pc5d_{y}.tif", nam=co("pc5d"), k=K5, crs="EPSG:32648", he_so=HE5)
for ps in MAN["point_sets"]:
    ten = os.path.splitext(os.path.basename(ps["duong_dan"]))[0]
    if os.path.exists(f"{HF_DIR}/diem/{ten}_duong_cong_k5.json"):
        ps["duong_cong"] = f"diem/{ten}_duong_cong_k5.json"
MAN["curves"] = dict(feats=["NDVI", "MNDWI"], ky=HS.get("ky"), k=K5)
MAN["du_lieu_diem"] = sorted(os.path.relpath(p, HF_DIR) for p in glob.glob(f"{HF_DIR}/diem/*.parquet"))
MAN["phien_ban"] = 2; MAN["cap_nhat"] = time.strftime("%Y-%m-%d %H:%M")
MAN["ghi_chu"] = (MAN.get("ghi_chu") or "") + " | Bản 2: thêm PC1-5 (xám, đối xứng quanh 0), dữ liệu 5 PC (đường mùa vụ " \
                  "ở điểm bất kỳ, k = 5), embedding (" + ", ".join(sorted({L['id'] for L in moi if L['kieu'] == 'rgb'})) + \
                  ") và embedding 64 chiều tại hai bộ điểm."
json.dump(MAN, open(f"{HF_DIR}/manifest.json", "w"), ensure_ascii=False, indent=1)
shutil.copyfile(f"{MOD}/LAY_MAU_DA_NAM.html", f"{HF_DIR}/LAY_MAU_DA_NAM.html")

rd = open(f"{HF_DIR}/README.md", encoding="utf-8").read().split("\n## Bản 2")[0]
rd += f'''
## Bản 2 ({time.strftime("%Y-%m-%d")})
- `pc1` ... `pc5`: từng PC của PCA chuỗi năm Sentinel-2 (mã {MA}), 8 bit, kéo giãn cố định đối xứng quanh 0 (PC × 100: ± {", ".join(str(round(v)) for v in KG5)}).
  Chỉ giữ 5 PC đầu: các PC sau mang mây, bóng mây và vệt quỹ đạo.
- `pc5d`: 5 PC (int16, × 100) + NFILL trên lưới UTM 48N gốc, để tái dựng đường NDVI, MNDWI 6 kỳ ở điểm bất kỳ.
- `diem/*_duong_cong_k5.json`: đường mùa vụ tính sẵn cho hai bộ điểm với k = 5 (bản k = 11 cũ vẫn giữ).
- Embedding: {", ".join(sorted({L['id'] for L in moi if L['kieu'] == 'rgb'})) or "(chưa có)"}; phép chiếu ở `emb/*_phep_chieu.json`.
- `diem/emb_*_*.parquet` (64 chiều) và `diem/pc5_*.parquet`: đặc trưng tại hai bộ điểm, mọi năm.
'''
open(f"{HF_DIR}/README.md", "w", encoding="utf-8").write(rd)
print("manifest bản 2:", [L["id"] for L in MAN["layers"]], "| pc:", MAN.get("pc", {}).get("duong_dan"))
print("dữ liệu tại điểm:", MAN["du_lieu_diem"])

In [ ]:
# ============ 5. ĐẨY CÁC TỆP MỚI LÊN HUGGING FACE ============
MAU = ["manifest.json", "manifest_v1.json", "README.md", "LAY_MAU_DA_NAM.html", "diem/*", "emb/*", "pc5d/*"] + \
      [f"pc{j}/*" for j in range(1, K5 + 1)] + [c["ten"] + s + "/*" for c in EMB_CAU_HINH for s in ("", "b")]
if GUI_LEN:
    from huggingface_hub import HfApi
    try:
        from google.colab import userdata; tok = userdata.get("HF_TOKEN")
    except Exception:
        import getpass; tok = getpass.getpass("HF token (quyền ghi): ")
    api = HfApi(token=tok)
    try:
        api.upload_large_folder(folder_path=HF_DIR, repo_id=HF_REPO, repo_type="dataset", allow_patterns=MAU)
    except (AttributeError, TypeError):
        api.upload_folder(folder_path=HF_DIR, repo_id=HF_REPO, repo_type="dataset", allow_patterns=MAU,
                          commit_message="Bản 2: PC1-5, dữ liệu 5 PC, embedding, đặc trưng tại điểm")
    if CAP_NHAT_SPACE:
        try:
            api.upload_file(path_or_fileobj=f"{HF_DIR}/LAY_MAU_DA_NAM.html", path_in_repo="index.html",
                            repo_id=SPACE, repo_type="space")
            print("đã cập nhật Space:", f"https://huggingface.co/spaces/{SPACE}")
        except Exception as e:
            print("không cập nhật được Space:", e)
    files_ = api.list_repo_files(HF_REPO, repo_type="dataset")
    print("xong:", f"https://huggingface.co/datasets/{HF_REPO}", "|", len(files_), "tệp trên kho")
    thieu_ = [os.path.relpath(p, HF_DIR) for pat in MAU for p in glob.glob(f"{HF_DIR}/{pat}")
              if os.path.isfile(p) and os.path.relpath(p, HF_DIR) not in files_]
    print("còn thiếu trên kho:", thieu_ or "không")
else:
    print("GUI_LEN = False: chưa đẩy. Mẫu tệp sẽ đẩy:", MAU)